# 07 — Training

Multi-source domain training of MS-ZeroGAD.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os
PROJECT_ROOT = '/content/drive/MyDrive/Project_GraphML/ms-zerogad'
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

In [ ]:
import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['PYTHONHASHSEED'] = '42'

import torch
import yaml
import json
from datetime import datetime

from ms_zerogad.data.loader import load_graph_dataset
from ms_zerogad.data.preprocessing import sparse_to_torch_dense, feature_to_torch
from ms_zerogad.pipeline.multipass import MultiPassPipeline
from ms_zerogad.training.train import train
from ms_zerogad.evaluation.metrics import evaluate_pipeline, evaluate_per_pass

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

from ms_zerogad.utils.seeds import set_all_seeds
set_all_seeds(42)

## Load configuration

In [ ]:
with open('configs/default.yaml') as f:
    cfg = yaml.safe_load(f)

# Set random seed
# torch.manual_seed(cfg['training']['seed'])

print(json.dumps(cfg, indent=2))

## Load source datasets (training)

In [ ]:
source_names = cfg['datasets']['source']
source_graphs = []

for name in source_names:
    path = f'/content/drive/MyDrive/Project_GraphML/ms-zerogad/ms_zerogad/data/raw/{name}.mat'
    if not os.path.exists(path):
        print(f'SKIP: {name} not found at {path}')
        continue

    A_sp, X_sp, y = load_graph_dataset(path)
    A = sparse_to_torch_dense(A_sp)
    X = feature_to_torch(X_sp, dense=True)

    source_graphs.append((X, A))
    print(f'Loaded {name}: n={X.shape[0]}, f={X.shape[1]}')

print(f'\nTotal source graphs: {len(source_graphs)}')

## Load target datasets (for periodic evaluation during training)

In [ ]:
target_names = cfg['datasets']['target']
target_graphs = {}

# Pick one small target graph for in-loop evaluation (faster than all 8)
for name in ['Cora']:  # use Cora for in-loop eval
    path = f'/content/drive/MyDrive/Project_GraphML/ms-zerogad/ms_zerogad/data/raw/{name}.mat'
    if not os.path.exists(path):
        print(f'SKIP: {name} not found at {path}')
        continue

    A_sp, X_sp, y_np = load_graph_dataset(path)
    A = sparse_to_torch_dense(A_sp)
    X = feature_to_torch(X_sp, dense=True)
    y = torch.from_numpy(y_np).long()

    target_graphs[name] = (X, A, y)
    print(f'Loaded eval target {name}: n={X.shape[0]}, anomalies={y.sum().item()}')

## Build pipeline

In [ ]:
pipeline = MultiPassPipeline(
    # Module 1
    d_prime=cfg['module1']['d_prime'],
    band_low=cfg['module1']['band_low'],
    band_high=cfg['module1']['band_high'],
    alpha_low=cfg['module1']['alpha_low'],
    alpha_mid=cfg['module1']['alpha_mid'],
    alpha_high=cfg['module1']['alpha_high'],
    adaptive_bands=cfg['module1'].get('adaptive_bands', False),               # ← thêm
    band_low_percentile=cfg['module1'].get('band_low_percentile', 0.33),      # ← thêm
    band_high_percentile=cfg['module1'].get('band_high_percentile', 0.67),
    # Module 2
    k_smoothing=cfg['module2']['k_smoothing'],
    sigma=cfg['module2']['sigma'],
    D_rff=cfg['module2']['D_rff'],
    d_svd=cfg['module2']['d_svd'],
    tau=cfg['module2']['tau'],
    kmeans_max_iter=cfg['module2']['kmeans_max_iter'],
    # Module 4
    d_hidden=cfg['module4']['d_hidden'],
    d_latent=cfg['module4']['d_latent'],
    num_encoder_layers=cfg['module4']['num_encoder_layers'],
    num_decoder_layers=cfg['module4']['num_decoder_layers'],
    dropout=cfg['module4']['dropout'],
).to(device)

n_params = sum(p.numel() for p in pipeline.parameters() if p.requires_grad)
print(f'Pipeline created. Trainable parameters: {n_params:,}')

## Define eval callback

In [ ]:
def eval_callback(pipeline, epoch):
    results = {}
    for name, (X, A, y) in target_graphs.items():
        m = evaluate_pipeline(pipeline, X, A, y, device=device)
        results[f'{name}_auroc'] = m['auroc']
        results[f'{name}_auprc'] = m['auprc']
    return results

## Train

In [ ]:
# os.makedirs(os.path.dirname(save_best_path), exist_ok=True)
history = train(
    pipeline,
    source_graphs,
    num_epochs=cfg['training']['num_epochs'],
    lr=cfg['training']['lr'],
    weight_decay=cfg['training']['weight_decay'],
    alpha=cfg['loss']['alpha'],
    beta=cfg['loss']['beta'],
    loss_weights=cfg['loss']['loss_weights'],
    grad_clip_norm=cfg['training']['grad_clip_norm'],
    cache_unified=True,
    eval_callback=eval_callback,
    eval_interval=cfg['training']['eval_interval'],
    save_best_path='checkpoints/best.pt',     # ← mới
    cfg_for_save=cfg,                           # ← mới
    device=device,
    verbose=True,
)

## Plot training curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Loss
ax = axes[0]
ax.plot(history['epoch'], history['total_loss'], label='Total', linewidth=2)
ax.plot(history['epoch'], history['pass1_loss'], '--', label='Pass 1', alpha=0.7)
ax.plot(history['epoch'], history['pass2_loss'], '--', label='Pass 2', alpha=0.7)
ax.plot(history['epoch'], history['pass3_loss'], '--', label='Pass 3', alpha=0.7)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Training Loss')
ax.legend()
ax.grid(alpha=0.3)

# Eval AUROC
ax = axes[1]
if history['eval']:
    epochs_eval = [e['epoch'] for e in history['eval']]
    for name in target_graphs:
        key = f'{name}_auroc'
        values = [e[key] for e in history['eval']]
        ax.plot(epochs_eval, values, '-o', label=name)
ax.set_xlabel('Epoch')
ax.set_ylabel('AUROC')
ax.set_title('Validation AUROC')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Save checkpoint

In [ ]:
# Safety check trước khi save
assert len(history['epoch']) > 0, "history is empty — model wasn't trained!"
assert history['total_loss'][-1] < history['total_loss'][0], \
    f"Loss didn't decrease: {history['total_loss'][0]} → {history['total_loss'][-1]}"

# Kiểm tra weights không phải initial state
total_grad_steps = len(history['epoch'])
print(f"Saving checkpoint after {total_grad_steps} epochs")
print(f"Loss: {history['total_loss'][0]:.4f} → {history['total_loss'][-1]:.4f}")

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
ckpt_dir = 'checkpoints'
os.makedirs(ckpt_dir, exist_ok=True)
ckpt_path = os.path.join(ckpt_dir, f'ms_zerogad_{timestamp}.pt')

torch.save({
    'state_dict': pipeline.state_dict(),
    'config': cfg,
    'history': history,
}, ckpt_path)

print(f'Saved checkpoint: {ckpt_path}')
print(f'File size: {os.path.getsize(ckpt_path) / 1024:.1f} KB')